In [1]:
import pandas as pd
import numpy as np
import os
from itertools import combinations
from scipy.stats import mannwhitneyu, kruskal, shapiro
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns

# Wczytanie danych z plików

In [2]:
print("="*50)
print("wczytywanie danych")
print("="*50)

models = ['autoencoder', 'cmae', 'masked_autoencoder']
classes = [2, 3, 4, 5, 10, 20, 50]

rows = []
file_count = 0

def find_csv_file(directory):

    if not os.path.exists(directory):
        return None
    csv_files = [f for f in os.listdir(directory) if f.endswith('.csv')]
    if csv_files:
        return os.path.join(directory, csv_files[0])
    return None




# ------- wczytywanie danych dla porównania modeli

print("\nWczytywanie danych dla analizy 1")

for model in models:
    for n_classes in classes:

        folder_name = f'{n_classes}_classes'
        base_dir = os.path.join(os.getcwd(),'..', '..', 'experiments', model, 'cifar100', folder_name)

        csv_path = find_csv_file(base_dir)

        if csv_path is None:
            print(f"Brak pliku w: {base_dir}")
            continue

        try:
            df = pd.read_csv(csv_path)

            df["model"] = model
            df["n_classes"] = n_classes
            df["weight"] = None  # bez wag dla podstawowych modeli

            rows.append(df)
            file_count += 1
            # print(f"Wczytano: {model}/{n_classes} - wierszy: {len(df)}")

        except Exception as e:
            print(f"Błąd przy wczytywaniu {csv_path}: {e}")


print("\nWczytywanie danych dla analizy 2 (wagi CMAE)...")
weight_folders = {
    2: {
        '0.1': '2_classes_waga_01',
        '0.5': '2_classes_waga_05',
        '1.0': '2_classes'
    },
    5: {
        '0.1': '5_classes_waga_01',
        '0.5': '5_classes_waga_05',
        '1.0': '5_classes'
    },
    20: {
        '0.1': '20_classes_waga_01',
        '0.5': '20_classes_waga_05',
        '1.0': '20_classes'
    }
}

for n_classes in [2, 5, 20]:
    for weight, folder_name in weight_folders[n_classes].items():
        base_dir = os.path.join(os.getcwd(),'..', '..', 'experiments', 'cmae', 'cifar100', folder_name)

        csv_path = find_csv_file(base_dir)

        if csv_path is None:
            print(f"Brak pliku  w: {base_dir}")
            continue
        try:
            df = pd.read_csv(csv_path)

            df["model"] = f"cmae_{weight}"  # cmae_0.1, cmae_0.5, cmae_1.0
            df["n_classes"] = n_classes
            df["weight"] = weight

            rows.append(df)
            file_count += 1
            # print(f"cmae (waga {weight})/{n_classes} - wierszy: {len(df)}")

        except Exception as e:
            print(f"Błąd przy wczytywaniu {csv_path}: {e}")


# Łączenie wszytsk ow jeden df
if not rows:
    print("\nBłąd nie wczytano plików.")
    exit(1)

all_df = pd.concat(rows, ignore_index=True)
# print(all_df[(all_df["model"] == "cmae_1.0") &(all_df["n_classes"] == 20)])
# print(all_df.columns)

print(f"\n{'='*50}")
print(f"Wczytano {file_count} plików")

# print(f"Modele: {sorted(all_df['model'].unique())}")
# print(f"iczby klas: {sorted(all_df['n_classes'].unique())}")
print(f"{'='*50}\n")

metrics = ["nmi", "ari", "silhouette", "dbi"]


wczytywanie danych

Wczytywanie danych dla analizy 1

Wczytywanie danych dla analizy 2 (wagi CMAE)...

Wczytano 30 plików



In [3]:


def test_normality(df, metric):
    """Test normalności """
    results = []

    for n_cls, sub in df.groupby('n_classes'):
        for model in sorted(sub['model'].unique()):
            values = sub[sub['model']==model][metric].dropna().values

            try:
                stat, p = shapiro(values)
                results.append({
                    'metric': metric,
                    'n_classes': n_cls,
                    'model': model,
                    'W_stat': stat,
                    'p_value': p,
                    'normal': 'Tak' if p > 0.05 else 'Nie'
                })
            except:
                continue

    return pd.DataFrame(results)



def run_pairwise_tests(df, metric):
    """Testy parami Mann-Whitney U + poprawka Bonferroniego"""
    results = []

    for n_cls, sub in df.groupby('n_classes'):
        models_list = sorted(sub['model'].unique())
        if len(models_list) < 2:
            continue

        values = {m: sub[sub['model']==m][metric].dropna().values for m in models_list}
        values = {k: v for k, v in values.items() if len(v) > 1}

        p_vals = []
        for m1, m2 in combinations(values.keys(), 2):
            try:
                u, p = mannwhitneyu(values[m1], values[m2], alternative='two-sided')

                results.append({
                    'metric': metric,
                    'n_classes': n_cls,
                    'model1': m1,
                    'model2': m2,
                    'mean1': values[m1].mean(),
                    'mean2': values[m2].mean(),
                    'p_raw': p
                })
                p_vals.append(p)
            except:
                continue

        if p_vals:
            _, p_corr, _, _ = multipletests(p_vals, alpha=0.05, method='bonferroni')
            for i, p in enumerate(p_corr):
                results[-len(p_corr)+i]['p_bonf'] = p
                results[-len(p_corr)+i]['Istotne'] = 'Tak' if p < 0.05 else 'Nie'

    return pd.DataFrame(results)

def run_kruskal_tests(df, metric):
    """Test Kruskal-Wallis dla modeli razem"""
    results = []

    for n_cls, sub in df.groupby('n_classes'):
        models_list = sorted(sub['model'].unique())
        if len(models_list) < 3:
            continue

        groups = [sub[sub['model']==m][metric].dropna().values for m in models_list]
        groups = [g for g in groups if len(g) > 0]

        if len(groups) >= 3:
            try:
                h, p = kruskal(*groups)
                results.append({
                    'metric': metric,
                    'n_classes': n_cls,
                    'H_stat': h,
                    'p_value': p,
                    'Istotny': 'Tak' if p < 0.05 else 'Nie'
                })
            except:
                continue

    return pd.DataFrame(results)


def create_boxplots(df, experiment_name, metrics):
    """Boxploty dla metryk"""

    n_metrics = len(metrics)
    n_cols = 2
    n_rows = (n_metrics + 1) // 2

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5*n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    axes = axes.flatten()

    for i, metric in enumerate(metrics):
        ax = axes[i]

        plot_data = df.copy()
        plot_data['n_classes'] = plot_data['n_classes'].astype(str)

        sns.boxplot(
            data=plot_data,
            x='n_classes',
            y=metric,
            hue='model',
            ax=ax,
            palette='Set2'
        )

        ax.set_title(f'{metric.upper()} - Porównanie modeli', fontsize=14, fontweight='bold')
        ax.set_xlabel('Liczba klas', fontsize=12)
        ax.set_ylabel(metric.upper(), fontsize=12)
        ax.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
        ax.grid(True, alpha=0.3, linestyle='--')

    # # Usuń puste subploty
    # for i in range(len(metrics), len(axes)):
    #     fig.delaxes(axes[i])

    plt.tight_layout()
    filename = f"{experiment_name}_boxplots.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    filename_eps = f"{experiment_name}_boxplots.eps"
    plt.savefig(filename_eps, dpi=300, bbox_inches='tight')
    plt.close()



In [4]:
def analyze_experiment(df, experiment_name, metrics):
    """Pełna analiza eksperymentu"""

    print(f"Analiza: {experiment_name}")


    print("Statystyki opisowe:")
    summary = df.groupby(['n_classes', 'model'])[metrics].agg(['mean', 'std', 'count'])
    print(summary)


    print(f"\n{'='*80}")
    print("Test normalności (Shapiro-Wilk):")

    normality_all = []
    for metric in metrics:
        result = test_normality(df, metric)
        if len(result) > 0:
            normality_all.append(result)

            # Podsumowanie
            normal_count = len(result[result['normal']=='Tak'])
            total_count = len(result)
            print(f"{metric.upper()}: {normal_count}/{total_count} grup ma rozkład normalny")

            # Pokaż nienormalne
            non_normal = result[result['normal']=='Nie']
            if len(non_normal) > 0:
                print(f"Nienormalne grupy:")
                for _, row in non_normal.iterrows():
                    print(f"    {row['model']}/{row['n_classes']} klas: p={row['p_value']:.4f}")

    # Zapisz normalność
    if normality_all:
        normality_df = pd.concat(normality_all, ignore_index=True)
        normality_df.to_csv(f"{experiment_name}_normality.csv", index=False)

    # Testy parami
    print(f"\n{'='*50}")
    print("Testy parami (Mann-Whitney U + Bonferroni):")
    pairwise_all = []
    for metric in metrics:
        print(f"\n{metric.upper()}:")
        result = run_pairwise_tests(df, metric)
        if len(result) > 0:
            pairwise_all.append(result)
            sig = result[result['Istotne']=='Tak']
            if len(sig) > 0:
                print(f"{len(sig)} istotnych różnic")
                for _, row in sig.iterrows():
                    better = row['model1'] if row['mean1'] > row['mean2'] else row['model2']
                    print(f"    {row['n_classes']} klas: {row['model1']} vs {row['model2']}: "
                          f"p={row['p_bonf']:.4f}")
            else:
                print("Brak istotnych różnic")


    print(f"\n{'='*50}")
    print("Testy Kruskal-Wallis")
    kruskal_all = []
    for metric in metrics:
        result = run_kruskal_tests(df, metric)
        if len(result) > 0:
            kruskal_all.append(result)
            print(f"\n{metric.upper()}:")
            for _, row in result.iterrows():
                print(f"  {row['n_classes']} klas: H={row['H_stat']:.3f}, "
                      f"p={row['p_value']:.4f} {row['Istotny']}")
        else:
            print(f"\n{metric.upper()}: brak wyników")


    if pairwise_all:
        pairwise_df = pd.concat(pairwise_all, ignore_index=True)
        pairwise_df.to_csv(f"{experiment_name}_pairwise.csv", index=False)
        print(f"\nZapisano: {experiment_name}_pairwise.csv")

    if kruskal_all:
        kruskal_df = pd.concat(kruskal_all, ignore_index=True)
        kruskal_df.to_csv(f"{experiment_name}_kruskal.csv", index=False)
        print(f"Zapisano: {experiment_name}_kruskal.csv")
    create_boxplots(df, experiment_name, metrics)


In [6]:

df_exp1 = all_df[
    (all_df['model'].isin(['autoencoder', 'cmae', 'masked_autoencoder'])) &
    (all_df['weight'].isna())
]

analyze_experiment(df_exp1, 'analysis_1_models', metrics)



df_exp2 = all_df[
    (all_df['model'].isin(['cmae_0.1', 'cmae_0.5', 'cmae_1.0'])) &
    (all_df['n_classes'].isin([2, 5, 20]))
]


analyze_experiment(df_exp2, 'analysis_2_weights', metrics)





Analiza: analysis_1_models
Statystyki opisowe:
                                   nmi                       ari            \
                                  mean       std count      mean       std   
n_classes model                                                              
2         autoencoder         0.088576  0.105255     5  0.109025  0.130203   
          cmae                0.049420  0.073021     5  0.040080  0.058738   
          masked_autoencoder  0.141321  0.058864     5  0.176301  0.077477   
3         autoencoder         0.204567  0.098179     5  0.208444  0.103361   
          cmae                0.114040  0.074977     5  0.110100  0.068659   
          masked_autoencoder  0.141983  0.042566     5  0.145577  0.052238   
4         autoencoder         0.185218  0.046704     5  0.156696  0.036451   
          cmae                0.112300  0.056562     5  0.087240  0.047713   
          masked_autoencoder  0.142522  0.099388     5  0.120431  0.090162   
5         autoenc

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Analiza: analysis_2_weights
Statystyki opisowe:
                         nmi                       ari                  \
                        mean       std count      mean       std count   
n_classes model                                                          
2         cmae_0.1  0.251150  0.220422     5  0.304540  0.253721     5   
          cmae_0.5  0.149744  0.245529     5  0.173822  0.283563     5   
          cmae_1.0  0.049420  0.073021     5  0.040080  0.058738     5   
5         cmae_0.1  0.248328  0.097683     5  0.209223  0.090651     5   
          cmae_0.5  0.146519  0.053555     5  0.104779  0.041915     5   
          cmae_1.0  0.145304  0.065729     5  0.114886  0.064294     5   
20        cmae_0.1  0.191166  0.021782     5  0.070483  0.012792     5   
          cmae_0.5  0.200128  0.023273     5  0.076048  0.016043     5   
          cmae_1.0  0.167284  0.032534     5  0.058490  0.014947     5   

                   silhouette                       dbi        

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
